## Setup

This notebook verifies the offset-learning workflow in simulation. The constants below set the model energy and the scan range used for each motor/diagnostic pair.

In [1]:
import math

import numpy as np
import pandas as pd
from bluesky import RunEngine

from lcls_beamline_toolbox.models.split_and_delay_motion import SND
from lcls_beamline_toolbox.models.split_and_delay_ophyd import SndOphyd
from lcls_beamline_toolbox.utility.bluesky_alignment import (
    DEFAULT_PHASE1_STEPS,
    DEFAULT_PHASE2_STEPS,
    scan_fit_center,
    scan_minimize_vertical_error,
)

ENERGY_EV = 9500
RNG_SEED = 7
START = -100e-6
STOP = 100e-6
STEPS = 81
SHOTS_PER_STEP = 1

## Offset Table



In [2]:
OFFSET_PV_BY_MOTOR = {
    "t1_th1": "XCS:USER:SND:T1_TH1_OFFSET",
    "t1_th2": "XCS:USER:SND:T1_TH2_OFFSET",
    "t2_th": "XCS:USER:SND:T2_TH_OFFSET",
    "t3_th": "XCS:USER:SND:T3_TH_OFFSET",
    "t4_th1": "XCS:USER:SND:T4_TH1_OFFSET",
    "t4_th2": "XCS:USER:SND:T4_TH2_OFFSET",
    "t1_chi1": "XCS:USER:SND:T1_CHI1_OFFSET",
    "t1_chi2": "XCS:USER:SND:T1_CHI2_OFFSET",
    "t4_chi1": "XCS:USER:SND:T4_CHI1_OFFSET",
    "t4_chi2": "XCS:USER:SND:T4_CHI2_OFFSET",
}


def build_phase_rows(phase_name, sequence):
    rows = []
    for label, motor, diagnostic in sequence:
        rows.append(
            {
                "phase": phase_name,
                "label": label,
                "motor": motor,
                "diagnostic": diagnostic,
                "offset_pv": OFFSET_PV_BY_MOTOR[motor],
                "model_value_at_alignment": None,
                "hardware_value_at_alignment": None,
                "learned_offset_native": None,
                "learned_offset_pv_units": None,
            }
        )
    return rows


offset_table = pd.DataFrame(
    build_phase_rows("phase1", DEFAULT_PHASE1_STEPS)
    + build_phase_rows("phase2", DEFAULT_PHASE2_STEPS)
)

## Model Reference

Here we create the aligned model and record the motor positions that correspond to alignment. These values become the reference points for the learned offsets: after the scan recovers the misaligned hardware position, we compare it back to this table.

In [3]:
snd = SND(ENERGY_EV)

for idx, row in offset_table.iterrows():
    offset_table.loc[idx, "model_value_at_alignment"] = snd.motor_dict[row["motor"]].wm()

offset_table

,phase,label,motor,diagnostic,offset_pv,model_value_at_alignment,hardware_value_at_alignment,learned_offset_native,learned_offset_pv_units
0,phase1,X1,t1_th1,t1_dh_sum,XCS:USER:SND:T1_TH1_OFFSET,0.346782,None,None,None
1,phase1,X2,t1_th2,dd_sum,XCS:USER:SND:T1_TH2_OFFSET,0.346782,None,None,None
2,phase1,X3,t4_th2,t4_dh_sum,XCS:USER:SND:T4_TH2_OFFSET,0.346782,None,None,None
3,phase1,X4,t4_th1,do_sum,XCS:USER:SND:T4_TH1_OFFSET,0.346782,None,None,None
4,phase1,CC1,t2_th,dcc_sum,XCS:USER:SND:T2_TH_OFFSET,0,None,None,None
5,phase1,CC2,t3_th,dco_sum,XCS:USER:SND:T3_TH_OFFSET,0,None,None,None
6,phase2,X1,t1_chi1,IP_cy,XCS:USER:SND:T1_CHI1_OFFSET,0,None,None,None
7,phase2,X2,t1_chi2,IP_cy,XCS:USER:SND:T1_CHI2_OFFSET,0,None,None,None
8,phase2,X3,t4_chi2,IP_cy,XCS:USER:SND:T4_CHI2_OFFSET,0,None,None,None
9,phase2,X4,t4_chi1,IP_cy,XCS:USER:SND:T4_CHI1_OFFSET,0,None,None,None


## Simulated Hardware

To test the procedure without touching real devices, we make a second `SND` model and randomly perturb each alignment motor. Wrapping this model with `SndOphyd` gives Bluesky something hardware-like to scan while still keeping the expected answer known.

In [4]:
rng = np.random.default_rng(RNG_SEED)

snd_hw = SND(ENERGY_EV)
for motor_name in offset_table["motor"]:
    delta = rng.uniform(-75e-6, 75e-6)
    snd_hw.motor_dict[motor_name].mvr(delta)

snd_hw.propagate_delay()
snd_hw.propagate_cc()

hw_backend = SndOphyd(snd_hw)

misalignment_summary = pd.DataFrame(
    {
        "motor": offset_table["motor"],
        "model_value": [snd.motor_dict[m].wm() for m in offset_table["motor"]],
        "hw_start_value": [snd_hw.motor_dict[m].wm() for m in offset_table["motor"]],
    }
)
misalignment_summary["start_delta"] = (
    misalignment_summary["hw_start_value"] - misalignment_summary["model_value"]
)

misalignment_summary

/Users/aminelamouchi/Desktop/archive/lcls_beamline_toolbox/lcls_beamline_toolbox/xraywavetrace/optics1d.py:7303: RuntimeWarning: invalid value encountered in scalar divide
  sx = np.sqrt(np.sum(norm_x * (self.x - cx) ** 2) / np.sum(norm_x)) * 1e6
/Users/aminelamouchi/Desktop/archive/lcls_beamline_toolbox/lcls_beamline_toolbox/xraywavetrace/optics1d.py:7304: RuntimeWarning: invalid value encountered in scalar divide
  sy = np.sqrt(np.sum(norm_y * (self.y - cy) ** 2) / np.sum(norm_y)) * 1e6


,motor,model_value,hw_start_value,start_delta
0,t1_th1,0.346782,0.346801,0.000019
1,t1_th2,0.346782,0.346841,0.000060
2,t4_th2,0.346782,0.346823,0.000041
3,t4_th1,0.346782,0.346741,-0.000041
4,t2_th,0.000000,-0.000030,-0.000030
5,t3_th,0.000000,0.000056,0.000056
6,t1_chi1,0.000000,-0.000074,-0.000074
7,t1_chi2,0.000000,0.000048,0.000048
8,t4_chi2,0.000000,0.000045,0.000045
9,t4_chi1,0.000000,-0.000005,-0.000005


## Learn Offsets

This is the main test of the method. For each row in the table, we run the same alignment scan we would use on hardware, compare the recovered hardware position to the aligned model position, and convert that difference into the units expected by the offset PV.

The printed `caput` lines are intentionally commented out so this stays a dry run until we are ready to write values.

In [10]:
RE_hw = RunEngine({})

for idx, row in offset_table.iterrows():
    motor = getattr(hw_backend, row["motor"])
    detector = getattr(hw_backend, row["diagnostic"])

    if row["phase"] == "phase1":
        result = scan_fit_center(
            RE_hw,
            motor,
            detector,
            start=START,
            stop=STOP,
            steps=STEPS,
            shots_per_step=SHOTS_PER_STEP,
            move=True,
        )
        hardware_value = float(result["center"])
    else:
        result = scan_minimize_vertical_error(
            RE_hw,
            motor,
            detector,
            start=START,
            stop=STOP,
            steps=STEPS,
            shots_per_step=SHOTS_PER_STEP,
            move=True,
        )
        hardware_value = float(result["best_position"])

    model_value = float(row["model_value_at_alignment"])
    learned_offset = hardware_value - model_value

    if "CHI" in row["offset_pv"]:
        learned_offset_pv_units = math.degrees(learned_offset) * 1e3
    else:
        learned_offset_pv_units = math.degrees(learned_offset)

    offset_table.loc[idx, "hardware_value_at_alignment"] = hardware_value
    offset_table.loc[idx, "learned_offset_native"] = learned_offset
    offset_table.loc[idx, "learned_offset_pv_units"] = learned_offset_pv_units

    print(f"# caput {row['offset_pv']} {learned_offset_pv_units}")

offset_table

/Users/aminelamouchi/Desktop/archive/lcls_beamline_toolbox/lcls_beamline_toolbox/xraywavetrace/optics1d.py:7303: RuntimeWarning: invalid value encountered in scalar divide
  sx = np.sqrt(np.sum(norm_x * (self.x - cx) ** 2) / np.sum(norm_x)) * 1e6
/Users/aminelamouchi/Desktop/archive/lcls_beamline_toolbox/lcls_beamline_toolbox/xraywavetrace/optics1d.py:7304: RuntimeWarning: invalid value encountered in scalar divide
  sy = np.sqrt(np.sum(norm_y * (self.y - cy) ** 2) / np.sum(norm_y)) * 1e6


# caput XCS:USER:SND:T1_TH1_OFFSET 2.9889079342770433e-05
# caput XCS:USER:SND:T1_TH2_OFFSET 3.086489200760781e-05
# caput XCS:USER:SND:T4_TH2_OFFSET 2.7503688801554687e-05
# caput XCS:USER:SND:T4_TH1_OFFSET 2.6097177126194755e-05
# caput XCS:USER:SND:T2_TH_OFFSET -0.0021948091088287856


/Users/aminelamouchi/opt/anaconda3/envs/adaexam/lib/python3.9/site-packages/scipy/optimize/_minpack_py.py:881: OptimizeWarning: Covariance of the parameters could not be estimated
  warnings.warn('Covariance of the parameters could not be estimated',


# caput XCS:USER:SND:T3_TH_OFFSET 0.002005745164225689
# caput XCS:USER:SND:T1_CHI1_OFFSET 1.4776464472458182
# caput XCS:USER:SND:T1_CHI2_OFFSET -1.3931891197491983
# caput XCS:USER:SND:T4_CHI2_OFFSET 2.5531236734781833
# caput XCS:USER:SND:T4_CHI1_OFFSET -0.27557878079141457


,phase,label,motor,diagnostic,offset_pv,model_value_at_alignment,hardware_value_at_alignment,learned_offset_native,learned_offset_pv_units
0,phase1,X1,t1_th1,t1_dh_sum,XCS:USER:SND:T1_TH1_OFFSET,0.346782,0.346782,0.000001,0.00003
1,phase1,X2,t1_th2,dd_sum,XCS:USER:SND:T1_TH2_OFFSET,0.346782,0.346782,0.000001,0.000031
2,phase1,X3,t4_th2,t4_dh_sum,XCS:USER:SND:T4_TH2_OFFSET,0.346782,0.346782,0.0,0.000028
3,phase1,X4,t4_th1,do_sum,XCS:USER:SND:T4_TH1_OFFSET,0.346782,0.346782,0.0,0.000026
4,phase1,CC1,t2_th,dcc_sum,XCS:USER:SND:T2_TH_OFFSET,0,-0.000038,-0.000038,-0.002195
5,phase1,CC2,t3_th,dco_sum,XCS:USER:SND:T3_TH_OFFSET,0,0.000035,0.000035,0.002006
6,phase2,X1,t1_chi1,IP_cy,XCS:USER:SND:T1_CHI1_OFFSET,0,0.000026,0.000026,1.477646
7,phase2,X2,t1_chi2,IP_cy,XCS:USER:SND:T1_CHI2_OFFSET,0,-0.000024,-0.000024,-1.393189
8,phase2,X3,t4_chi2,IP_cy,XCS:USER:SND:T4_CHI2_OFFSET,0,0.000045,0.000045,2.553124
9,phase2,X4,t4_chi1,IP_cy,XCS:USER:SND:T4_CHI1_OFFSET,0,-0.000005,-0.000005,-0.275579


In [ ]:
import numpy as np
from skimage.filters import threshold_otsu

class EventProcessor:
    def begin_run(self): pass
    def end_run(self): pass
    def begin_step(self, step): pass
    def end_step(self, step): pass
    
    def on_event(self, In, *args, **kwargs):
        img = np.asarray(In, dtype=np.float32)
        thr = threshold_otsu(img)
        return np.where(img > thr, img, 0.0).astype(np.float32)